# Python for Biological Modeling and Automation Workflow

This notebook scaffold mirrors the Python-first workflow: model validation, logistic growth, compartment modeling, scenario sweeps, sensitivity summaries, and provenance documentation.

In [ ]:
from pathlib import Path
import pandas as pd

article_dir = Path.cwd().parent
logistic = pd.read_csv(article_dir / 'data' / 'logistic_parameters.csv')
compartment = pd.read_csv(article_dir / 'data' / 'compartment_parameters.csv')
logistic.head()

In [ ]:
def simulate_logistic(initial_population, growth_rate, carrying_capacity, dt, steps):
    population = float(initial_population)
    rows = []
    for step in range(steps + 1):
        rows.append({'step': step, 'time': step * dt, 'population': population})
        growth = growth_rate * population * (1 - population / carrying_capacity)
        population = max(population + dt * growth, 0.0)
    return pd.DataFrame(rows)

row = logistic[logistic['scenario'] == 'baseline'].iloc[0]
trajectory = simulate_logistic(row['initial_population'], row['growth_rate'], row['carrying_capacity'], row['dt'], int(row['steps']))
trajectory.tail().round(5)

In [ ]:
def simulate_two_compartment(initial_a, initial_b, k_ab, k_ba, k_clear, dt, steps):
    amount_a = float(initial_a)
    amount_b = float(initial_b)
    rows = []
    for step in range(steps + 1):
        rows.append({'step': step, 'time': step * dt, 'compartment_a': amount_a, 'compartment_b': amount_b, 'total_amount': amount_a + amount_b})
        flow_ab = k_ab * amount_a
        flow_ba = k_ba * amount_b
        clearance = k_clear * amount_a
        amount_a = max(amount_a + dt * (-flow_ab + flow_ba - clearance), 0.0)
        amount_b = max(amount_b + dt * (flow_ab - flow_ba), 0.0)
    return pd.DataFrame(rows)

row = compartment[compartment['scenario'] == 'baseline_exchange'].iloc[0]
trajectory = simulate_two_compartment(row['initial_a'], row['initial_b'], row['k_ab'], row['k_ba'], row['k_clear'], row['dt'], int(row['steps']))
trajectory.tail().round(5)